# Training classifiers with MultiTrain

In this walkthrough, we will train MultiTrain's complete classifier catalog on the Titanic dataset included with the repository.

## Import MultiTrain and load the dataset

In [1]:
from pathlib import Path

import pandas as pd

import MultiTrain
from MultiTrain import MultiClassifier

print(f"MultiTrain version: {MultiTrain.__version__}")

dataset_path = Path("examples/datasets/train.csv")
df = pd.read_csv(dataset_path)
df.head()

MultiTrain version: 1.2.0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Take a quick look at the data

`Survived` is the target. `Age` and `Embarked` contain missing values, while several columns contain text that must either be encoded or removed before the models can use them.

In [2]:
print(f"Dataset shape: {df.shape}")
print()
print("Target distribution:")
print(df["Survived"].value_counts().sort_index())
print()
print("Columns containing missing values:")
print(df.isna().sum()[df.isna().any()])

Dataset shape: (891, 12)

Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64

Columns containing missing values:
Age         177
Cabin       687
Embarked      2
dtype: int64


## Configure MultiTrain

Each estimator receives one CPU thread and MultiTrain may train two different estimators at the same time. `custom_models` is left as `None`, which means this run includes every built-in classifier. GPU training is intentionally left off so the notebook runs on an ordinary laptop and in CI.

In [3]:
train = MultiClassifier(
    n_jobs=1,
    model_workers=2,
    random_state=42,
    max_iter=300,
)

## Prepare the train and test sets

The passenger name, ticket, cabin, and identifier are high-cardinality identifiers rather than useful columns for this introductory example, so we drop them. MultiTrain fills the missing values after splitting the data and learns categorical encodings from the training partition only.

In [4]:
classification_split = train.split(
    data=df,
    target="Survived",
    test_size=0.2,
    random_state=42,
    auto_cat_encode=True,
    fix_nan_custom={"Age": "interpolate", "Embarked": "ffill"},
    drop=["PassengerId", "Name", "Ticket", "Cabin"],
)

X_train, X_test, y_train, y_test = classification_split
print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
X_train.head()

Training features: (712, 7)
Test features: (179, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
692,3,0,0.0,0,0,56.4958,0
481,2,0,0.0,0,0,0.0000,0
527,1,0,0.0,0,0,221.7792,0
855,3,1,18.0,0,1,9.3500,0
801,2,1,31.0,1,1,26.2500,0


## Train and measure every classifier

`show_train_score=True` places the training and test measurements next to each other. Sorting by accuracy only changes the order of the table; every model is still present.

In [5]:
pd.set_option("display.max_rows", None)

classification_results = train.fit(
    datasplits=classification_split,
    show_train_score=True,
    sort="accuracy",
)

classification_results

Training Models:   0%|          | 0/27 [00:00<?, ?it/s]

,accuracy,precision_train,precision,recall_train,recall,balanced_accuracy_train,balanced_accuracy,accuracy_train,f1_train,f1,roc_auc_train,roc_auc,Time
MLPClassifier,0.815642,0.821577,0.821429,0.725275,0.666667,0.813662,0.787879,0.83427,0.770428,0.736,0.877707,0.857839,95.17ms
SVC,0.815642,0.901961,0.86,0.673993,0.623188,0.814217,0.779776,0.84691,0.771488,0.722689,0.887169,0.838208,66.10ms
DecisionTreeClassifier,0.810056,1.0,0.761194,0.970696,0.73913,0.985348,0.796838,0.988764,0.98513,0.75,0.999695,0.795652,6.01ms
NuSVC,0.810056,0.810127,0.818182,0.703297,0.652174,0.800396,0.780632,0.823034,0.752941,0.725806,0.867986,0.827668,76.55ms
RandomForestClassifier,0.804469,1.0,0.774194,0.970696,0.695652,0.985348,0.78419,0.988764,0.98513,0.732824,0.999378,0.829381,182.96ms
ExtraTreesClassifier,0.804469,1.0,0.75,0.970696,0.73913,0.985348,0.792292,0.988764,0.98513,0.744526,0.999695,0.832609,160.52ms
LinearSVC,0.804469,0.75,0.783333,0.703297,0.681159,0.778755,0.781489,0.796348,0.725898,0.728682,0.859642,0.837813,6.47ms
LogisticRegression,0.798883,0.754864,0.789474,0.710623,0.652174,0.783557,0.771542,0.800562,0.732075,0.714286,0.860476,0.841107,8.65ms
CatBoostClassifier,0.798883,0.915254,0.811321,0.791209,0.623188,0.872825,0.76614,0.891854,0.848723,0.704918,0.94941,0.857444,558.59ms
GradientBoostingClassifier,0.793296,0.92827,0.807692,0.805861,0.608696,0.883568,0.758893,0.901685,0.862745,0.694215,0.954997,0.828063,127.09ms


## Confirm the run

The final check counts the returned models and identifies models for which every test metric is missing. An individual missing metric does not necessarily mean fitting failed—for example, some estimators cannot provide the score needed for ROC AUC.

In [6]:
classification_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "balanced_accuracy",
]
failed_classifiers = classification_results[
    classification_metrics
].isna().all(axis=1)

print(f"Models returned: {len(classification_results)}")
print(f"Models with every test metric missing: {failed_classifiers.sum()}")
if failed_classifiers.any():
    print(classification_results.index[failed_classifiers].tolist())

Models returned: 27
Models with every test metric missing: 0


## Inspect the fitted models and their cached outputs

The dataframe returned by `fit` is also kept in `results_`. MultiTrain keeps the fitted estimator objects in `models_`, while `predictions_` and `probabilities_` contain the exact outputs that were used to calculate the measurements above. Because we asked for training scores, both the training and test predictions are available.

In [7]:
model_name = "LogisticRegression"
fitted_model = train.models_[model_name]

print(f"Stored result is the returned dataframe: {train.results_ is classification_results}")
print(f"Stored fitted object: {type(fitted_model).__name__}")
print(f"Training predictions: {train.predictions_['train'][model_name].shape}")
print(f"Test predictions: {train.predictions_['test'][model_name].shape}")
print(f"Test probabilities: {train.probabilities_['test'][model_name].shape}")

Stored result is the returned dataframe: True
Stored fitted object: Pipeline
Training predictions: (712,)
Test predictions: (179,)
Test probabilities: (179, 2)


## Use your own estimator names and parameters

Sometimes the built-in defaults are not the configuration you want to compare. A dictionary passed to `custom_models` lets you provide estimator objects and choose the names shown in the result table. `model_params` can then override parameters by using those same names. MultiTrain fits copies, so the two original objects below remain untouched.

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

source_tree = DecisionTreeClassifier(random_state=42)
source_svc = make_pipeline(
    StandardScaler(),
    SVC(probability=True, random_state=42),
)

configured_train = MultiClassifier(
    n_jobs=1,
    model_workers=2,
    custom_models={
        "small decision tree": source_tree,
        "scaled support vector machine": source_svc,
    },
    model_params={
        "small decision tree": {"max_depth": 4},
        "scaled support vector machine": {"svc__C": 0.5},
    },
)

configured_results = configured_train.fit(
    datasplits=classification_split,
    show_train_score=True,
    sort="accuracy",
)

configured_results

Training Models:   0%|          | 0/2 [00:00<?, ?it/s]

,accuracy,precision_train,precision,recall_train,recall,balanced_accuracy_train,balanced_accuracy,accuracy_train,f1_train,f1,roc_auc_train,roc_auc,Time
scaled support vector machine,0.821229,0.850877,0.849057,0.710623,0.652174,0.816587,0.789723,0.841292,0.774451,0.737705,0.875904,0.826877,94.00ms
small decision tree,0.782123,0.920213,0.857143,0.6337,0.521739,0.799766,0.733597,0.838483,0.750542,0.648649,0.887915,0.815679,4.49ms


## Check warnings and failures

Warnings and failures belong to the run that produced them. `warnings_` records the model, warning category, and message. `failures_` also records the execution stage and exception type. Empty tables here mean this configured run completed without either kind of diagnostic.

In [9]:
print("Fitted model names:")
print(list(configured_train.models_))
fitted_tree = configured_train.models_["small decision tree"]
fitted_svc = configured_train.models_["scaled support vector machine"]
print(f"Tree max_depth: {fitted_tree.max_depth}")
print(f"SVC C value: {fitted_svc.get_params()['svc__C']}")
print(f"Original tree was fitted: {hasattr(source_tree, 'tree_')}")
print()
print("Warnings:")
display(configured_train.warnings_)
print("Failures:")
display(configured_train.failures_)

Fitted model names:
['small decision tree', 'scaled support vector machine']
Tree max_depth: 4
SVC C value: 0.5
Original tree was fitted: False

Warnings:


,Model,Category,Message


Failures:


,Model,Stage,Exception,Message
